# PanDx Reproduction Notebook
**AI-assisted Early Detection of Pancreatic Ductal Adenocarcinoma on Contrast-enhanced CT**

Liu et al. (2025) — 1st place, PANORAMA Challenge
- Paper: https://arxiv.org/abs/2503.10068
- Code:  https://github.com/han-liu/PanDx

## Pipeline overview
```
CT scan (full resolution)
   │
   ▼
[Stage 1]  Downsample → nnU-Net (Dataset103) → Pancreas segmentation mask
   │
   ▼
[Stage 2]  Crop ROI (±100/50/15 mm) → nnU-Net (Dataset107, CE loss) → PDAC prob map
   │
   ▼
[Post]     Peak-scaled candidate extraction (α = 1/15) → patient likelihood score
```

## 0. Environment setup & imports

In [ ]:
import os
import os.path as osp
import json
import shutil
import subprocess
import time
import warnings
from glob import glob
from pathlib import Path

import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Add local packages to path
import sys
REPO_ROOT = osp.abspath('.')          # /Desktop/PanDx
sys.path.insert(0, osp.join(REPO_ROOT, 'packages', 'nnunetv2'))
sys.path.insert(0, osp.join(REPO_ROOT, 'packages', 'report-guided-annotation', 'src'))

from report_guided_annotation.extract_lesion_candidates import extract_lesion_candidates

print('Imports OK')
print('REPO_ROOT:', REPO_ROOT)

## 1. Configuration — set your paths here

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Raw PANORAMA dataset directory (contains .mha / .nii.gz images + clinical JSON)
INPUT_DIR   = osp.join(REPO_ROOT, 'workspace', 'test_example', 'input')

# Where all outputs (detection maps, likelihood scores) are written
OUTPUT_DIR  = osp.join(REPO_ROOT, 'workspace', 'test_example', 'output')

# nnU-Net model weights directory
MODEL_DIR   = osp.join(REPO_ROOT, 'workspace', 'nnUNet_results')

# Intermediate working folder (auto-cleaned at end)
WORKING_DIR = osp.join(OUTPUT_DIR, 'itm')

# ── Model identifiers ─────────────────────────────────────────────────────────
STAGE1_TASK       = 103
STAGE1_TRAINER    = 'nnUNetTrainer'
STAGE1_PLAN       = 'nnUNetPlans'

STAGE2_TASK       = 107
STAGE2_TRAINER    = 'nnUNetTrainerCELossLesionSplit'
STAGE2_PLAN       = 'nnUNetPlans_v3'

# ── Hyper-parameters ──────────────────────────────────────────────────────────
DOWNSAMPLE_SPACING = (4.5, 4.5, 9.0)   # mm — low-res spacing for Stage 1
ROI_MARGINS        = [100, 50, 15]      # mm — x/y/z padding around pancreas bbox
INV_ALPHA          = 15                 # τ = (1/15) · P(x*)  peak-scaling factor

for d in [OUTPUT_DIR, WORKING_DIR,
          osp.join(OUTPUT_DIR, 'pdac-detection-map')]:
    os.makedirs(d, exist_ok=True)

print('Config ready.')
print(f'  Input : {INPUT_DIR}')
print(f'  Output: {OUTPUT_DIR}')
print(f'  Models: {MODEL_DIR}')

## 2. Helper functions

In [ ]:
def get_file_extension(path: str) -> str:
    base, ext = osp.splitext(path)
    if ext == '.gz' and base.endswith('.nii'):
        return '.nii.gz'
    return ext


def resample_img(itk_image, out_spacing=(2.0, 2.0, 2.0), is_label=False,
                 out_size=None, out_origin=None, out_direction=None):
    """Resample an ITK image to the requested voxel spacing."""
    orig_spacing = itk_image.GetSpacing()
    orig_size    = itk_image.GetSize()
    if out_size is None:
        out_size = [
            int(np.round(orig_size[i] * orig_spacing[i] / out_spacing[i]))
            for i in range(3)
        ]
    rs = sitk.ResampleImageFilter()
    rs.SetOutputSpacing(out_spacing)
    rs.SetSize(out_size)
    rs.SetOutputDirection(out_direction or itk_image.GetDirection())
    rs.SetOutputOrigin(out_origin or itk_image.GetOrigin())
    rs.SetTransform(sitk.Transform())
    rs.SetDefaultPixelValue(itk_image.GetPixelIDValue())
    rs.SetInterpolator(sitk.sitkNearestNeighbor if is_label else sitk.sitkBSpline)
    return rs.Execute(itk_image)


def downsample_dataset(img_dir: str, save_dir: str,
                       spacing=DOWNSAMPLE_SPACING) -> None:
    """Downsample all images in img_dir to low-res spacing for Stage-1 inference."""
    os.makedirs(save_dir, exist_ok=True)
    img_paths = sorted(glob(osp.join(img_dir, '*.*')))
    assert img_paths, f'No images found in {img_dir}'
    for p in tqdm(img_paths, desc='Downsample'):
        ext  = get_file_extension(p)
        img  = sitk.ReadImage(p, sitk.sitkFloat32)
        resampled = resample_img(img, spacing)
        out_name  = osp.basename(p).replace(ext, '_0000.nii.gz')
        sitk.WriteImage(resampled, osp.join(save_dir, out_name))


def crop_roi(img_dir: str, low_mask_dir: str, save_dir: str,
             margins=ROI_MARGINS) -> dict:
    """
    Crop the high-resolution CT around the predicted pancreas bounding box.
    Returns a dict mapping case-id → crop coordinate slices.
    """
    os.makedirs(save_dir, exist_ok=True)
    img_paths = sorted(glob(osp.join(img_dir, '*.*')))
    crop_coords = {}
    for p in tqdm(img_paths, desc='Crop ROI'):
        ext          = get_file_extension(p)
        mask_path    = osp.join(low_mask_dir, osp.basename(p).replace(ext, '.nii.gz'))
        img          = sitk.ReadImage(p, sitk.sitkFloat32)
        low_mask     = sitk.ReadImage(mask_path)

        # Keep only pancreas label (class 1)
        mask_np = sitk.GetArrayFromImage(low_mask)
        mask_np = (mask_np == 1).astype(np.uint8)
        nz = np.nonzero(mask_np)
        min_x, max_x = int(nz[2].min()), int(nz[2].max())
        min_y, max_y = int(nz[1].min()), int(nz[1].max())
        min_z, max_z = int(nz[0].min()), int(nz[0].max())

        # Convert low-res indices → physical → full-res indices
        sp = low_mask.TransformIndexToPhysicalPoint
        ip = img.TransformPhysicalPointToIndex
        start_idx  = ip(sp((min_x, min_y, min_z)))
        finish_idx = ip(sp((max_x, max_y, max_z)))

        spacing = img.GetSpacing()
        size    = img.GetSize()
        mx = int(margins[0] / spacing[0])
        my = int(margins[1] / spacing[1])
        mz = int(margins[2] / spacing[2])

        xs = max(0, start_idx[0] - mx);  xf = min(size[0], finish_idx[0] + mx)
        ys = max(0, start_idx[1] - my);  yf = min(size[1], finish_idx[1] + my)
        zs = max(0, start_idx[2] - mz);  zf = min(size[2], finish_idx[2] + mz)

        cropped   = img[xs:xf, ys:yf, zs:zf]
        case_id   = osp.basename(p).replace(ext, '')
        crop_coords[case_id] = dict(x_start=xs, x_finish=xf,
                                    y_start=ys, y_finish=yf,
                                    z_start=zs, z_finish=zf)
        sitk.WriteImage(cropped,
                        osp.join(save_dir, osp.basename(p).replace(ext, '_0000.nii.gz')))
    return crop_coords


def run_nnunet_predict(model_dir: str, input_dir: str, output_dir: str,
                       task: int, trainer: str = 'nnUNetTrainer',
                       plan: str = 'nnUNetPlans',
                       configuration: str = '3d_fullres',
                       checkpoint: str = 'checkpoint_final.pth',
                       folds: str = '0,1,2,3,4',
                       save_probs: bool = True,
                       tta: bool = True) -> None:
    """Call nnUNetv2_predict as a subprocess."""
    os.environ['RESULTS_FOLDER'] = model_dir
    os.makedirs(output_dir, exist_ok=True)
    cmd = [
        'nnUNetv2_predict',
        '-d',  str(task),
        '-i',  input_dir,
        '-o',  output_dir,
        '-c',  configuration,
        '-tr', trainer,
        '-p',  plan,
        '--continue_prediction',
        '-f',  *folds.split(','),
        '-chk', checkpoint,
    ]
    if save_probs:
        cmd.append('--save_probabilities')
    if not tta:
        cmd.append('--disable_tta')
    print('Running:', ' '.join(str(c) for c in cmd))
    subprocess.check_call(cmd)


def postprocess_probmap(npz_path: str) -> np.ndarray:
    """Load .npz from nnU-Net and extract the PDAC (class 1) probability map."""
    data = np.load(npz_path)
    return data['probabilities'][1].astype(np.float32)


def build_full_size_detection_map(prob_map: np.ndarray,
                                  crop_coords: dict,
                                  reference_image: sitk.Image,
                                  inv_alpha: int = INV_ALPHA):
    """
    Apply peak-scaled lesion candidate extraction (τ = prob_map.max() / inv_alpha),
    then paste the result back into a full-size volume.

    Returns (detection_map_itk, patient_likelihood_score).
    """
    lesion_candidates, _, _ = extract_lesion_candidates(
        prob_map, dynamic_threshold_factor=inv_alpha
    )
    patient_score = float(np.max(lesion_candidates))

    full_shape = sitk.GetArrayFromImage(reference_image).shape
    full_map   = np.zeros(full_shape, dtype=np.float32)
    full_map[
        crop_coords['z_start']:crop_coords['z_finish'],
        crop_coords['y_start']:crop_coords['y_finish'],
        crop_coords['x_start']:crop_coords['x_finish'],
    ] = lesion_candidates

    out_itk = sitk.GetImageFromArray(full_map)
    out_itk.CopyInformation(reference_image)
    return out_itk, patient_score


def write_json(path: str, content: dict) -> None:
    with open(path, 'w') as f:
        json.dump(content, f, indent=4)


print('Helper functions defined.')

## 3. (Optional) Inspect the dataset
Summarise what images are in `INPUT_DIR` and optionally display one slice.

In [ ]:
img_paths = sorted([
    p for p in glob(osp.join(INPUT_DIR, '*.*'))
    if not p.endswith('.json') and not osp.isdir(p)
])
print(f'Found {len(img_paths)} image(s) in {INPUT_DIR}:')
for p in img_paths:
    img = sitk.ReadImage(p)
    print(f'  {osp.basename(p):40s}  size={img.GetSize()}  spacing={tuple(f"{s:.2f}" for s in img.GetSpacing())}')

In [ ]:
# Visualise a central axial slice of the first image
if img_paths:
    ex = sitk.ReadImage(img_paths[0], sitk.sitkFloat32)
    arr = sitk.GetArrayFromImage(ex)          # shape: (Z, Y, X)
    mid_z = arr.shape[0] // 2
    plt.figure(figsize=(6, 6))
    plt.imshow(arr[mid_z], cmap='gray', vmin=-200, vmax=300)
    plt.title(f'{osp.basename(img_paths[0])} — axial slice {mid_z}')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 4. Stage 1 — Pancreas localisation at low resolution

### 4a. Downsample to (4.5, 4.5, 9.0) mm

In [ ]:
LOW_IMAGE_DIR = osp.join(WORKING_DIR, 'LowImagesTr')
downsample_dataset(INPUT_DIR, LOW_IMAGE_DIR, DOWNSAMPLE_SPACING)
print('Downsampled images:', sorted(os.listdir(LOW_IMAGE_DIR)))

### 4b. nnU-Net inference — Dataset 103 (5-fold ensemble)

In [ ]:
LOW_PRED_DIR = osp.join(WORKING_DIR, 'LowPred')

run_nnunet_predict(
    model_dir     = MODEL_DIR,
    input_dir     = LOW_IMAGE_DIR,
    output_dir    = LOW_PRED_DIR,
    task          = STAGE1_TASK,
    trainer       = STAGE1_TRAINER,
    plan          = STAGE1_PLAN,
    folds         = '0,1,2,3,4',
    save_probs    = True,
    tta           = True,
)
print('Stage-1 predictions:', sorted(os.listdir(LOW_PRED_DIR)))

### 4c. Visualise Stage-1 segmentation overlay

In [ ]:
low_preds = sorted(glob(osp.join(LOW_PRED_DIR, '*.nii.gz')))
if low_preds:
    low_img_path  = sorted(glob(osp.join(LOW_IMAGE_DIR, '*.nii.gz')))[0]
    low_mask_path = low_preds[0]

    low_arr  = sitk.GetArrayFromImage(sitk.ReadImage(low_img_path, sitk.sitkFloat32))
    mask_arr = sitk.GetArrayFromImage(sitk.ReadImage(low_mask_path))

    # Find slice with most pancreas (class 1) voxels
    pancreas_slices = (mask_arr == 1).sum(axis=(1, 2))
    best_z = int(np.argmax(pancreas_slices))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(low_arr[best_z],  cmap='gray', vmin=-200, vmax=300)
    axes[0].set_title(f'Low-res CT — slice {best_z}')
    axes[0].axis('off')

    axes[1].imshow(low_arr[best_z],  cmap='gray', vmin=-200, vmax=300)
    axes[1].imshow(mask_arr[best_z] == 1, cmap='Reds',   alpha=0.4)  # pancreas
    axes[1].imshow(mask_arr[best_z] == 2, cmap='Blues',  alpha=0.4)  # duct
    axes[1].imshow(mask_arr[best_z] == 3, cmap='Greens', alpha=0.5)  # PDAC
    axes[1].set_title('Stage-1 segmentation overlay')
    axes[1].axis('off')

    patches = [
        mpatches.Patch(color='red',   alpha=0.6, label='Pancreas (1)'),
        mpatches.Patch(color='blue',  alpha=0.6, label='Duct (2)'),
        mpatches.Patch(color='green', alpha=0.6, label='PDAC (3)'),
    ]
    axes[1].legend(handles=patches, loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()

## 5. Stage 1 → Stage 2: Crop high-resolution ROI

Expand the predicted pancreas bounding box by **100 × 50 × 15 mm³** in x/y/z.

In [ ]:
CROPPED_IMAGE_DIR = osp.join(WORKING_DIR, 'CroppedImages')

crop_coordinates = crop_roi(
    img_dir      = INPUT_DIR,
    low_mask_dir = LOW_PRED_DIR,
    save_dir     = CROPPED_IMAGE_DIR,
    margins      = ROI_MARGINS,
)

print('\nCrop coordinates:')
for case_id, coords in crop_coordinates.items():
    print(f'  {case_id}: {coords}')

In [ ]:
# Visualise one cropped ROI vs the full scan
if img_paths:
    full_arr    = sitk.GetArrayFromImage(sitk.ReadImage(img_paths[0], sitk.sitkFloat32))
    cropped_arr = sitk.GetArrayFromImage(
        sitk.ReadImage(sorted(glob(osp.join(CROPPED_IMAGE_DIR, '*.nii.gz')))[0],
                       sitk.sitkFloat32)
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(full_arr[full_arr.shape[0]//2],    cmap='gray', vmin=-200, vmax=300)
    axes[0].set_title('Full CT (mid-axial)')
    axes[0].axis('off')
    axes[1].imshow(cropped_arr[cropped_arr.shape[0]//2], cmap='gray', vmin=-200, vmax=300)
    axes[1].set_title('Cropped ROI (mid-axial)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## 6. Stage 2 — Fine-scale PDAC detection

nnU-Net with **ResU-Net backbone** and **CE-only loss** (`nnUNetTrainerCELossLesionSplit`),  
trained on Dataset 107 (cropped pancreas ROIs).  
Softmax outputs are automatically averaged across all 5 folds during inference.

In [ ]:
CROPPED_PRED_DIR = osp.join(WORKING_DIR, 'CroppedPred')

run_nnunet_predict(
    model_dir     = MODEL_DIR,
    input_dir     = CROPPED_IMAGE_DIR,
    output_dir    = CROPPED_PRED_DIR,
    task          = STAGE2_TASK,
    trainer       = STAGE2_TRAINER,
    plan          = STAGE2_PLAN,
    folds         = '0,1,2,3,4',
    save_probs    = True,
    tta           = True,
)
print('Stage-2 predictions:', sorted(os.listdir(CROPPED_PRED_DIR)))

## 7. Post-processing — peak-scaled lesion candidate extraction

**Algorithm (from the paper):**
1. Set threshold τ = (1 / `inv_alpha`) × P(x*)  where x* is the voxel with maximum predicted probability.
2. Keep connected components above τ → lesion candidates.
3. Patient-level likelihood = max voxel probability in the candidate map.
4. Paste the candidate map back into a full-size volume aligned to the original CT.

In [ ]:
npz_fps = sorted(glob(osp.join(CROPPED_PRED_DIR, '*.npz')))
img_fps = sorted([
    p for p in glob(osp.join(INPUT_DIR, '*.*'))
    if not p.endswith('.json') and not osp.isdir(p)
])

assert len(npz_fps) == len(img_fps), (
    f'Mismatch: {len(npz_fps)} predictions vs {len(img_fps)} input images'
)

likelihoods = {}

for npz_fp, img_fp in zip(npz_fps, img_fps):
    ext      = get_file_extension(img_fp)
    case_id  = osp.basename(npz_fp)[:-4]          # strip .npz
    assert osp.basename(img_fp).replace(ext, '') == case_id, \
        f'File order mismatch: {osp.basename(img_fp)} vs {osp.basename(npz_fp)}'

    ref_img  = sitk.ReadImage(img_fp, sitk.sitkFloat32)
    prob_map = postprocess_probmap(npz_fp)

    det_map, score = build_full_size_detection_map(
        prob_map, crop_coordinates[case_id], ref_img, INV_ALPHA
    )

    out_path = osp.join(OUTPUT_DIR, 'pdac-detection-map', f'{case_id}.nii.gz')
    sitk.WriteImage(det_map, out_path)
    likelihoods[case_id] = score
    print(f'  {case_id:40s}  likelihood = {score:.4f}')

write_json(osp.join(OUTPUT_DIR, 'pdac-likelihood.json'), likelihoods)
print(f'\nLikelihood scores saved to: {osp.join(OUTPUT_DIR, "pdac-likelihood.json")}')

## 8. Visualise detection maps

In [ ]:
det_map_paths = sorted(glob(osp.join(OUTPUT_DIR, 'pdac-detection-map', '*.nii.gz')))

for det_path, img_fp in zip(det_map_paths, img_fps):
    ct_arr  = sitk.GetArrayFromImage(sitk.ReadImage(img_fp,   sitk.sitkFloat32))
    det_arr = sitk.GetArrayFromImage(sitk.ReadImage(det_path, sitk.sitkFloat32))
    case_id = osp.basename(det_path).replace('.nii.gz', '')

    # Slice with highest detection signal
    best_z  = int(np.argmax(det_arr.max(axis=(1, 2))))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(ct_arr[best_z],  cmap='gray',   vmin=-200, vmax=300)
    axes[0].set_title('CT (axial)')
    axes[0].axis('off')

    axes[1].imshow(det_arr[best_z], cmap='hot',    vmin=0,    vmax=1)
    axes[1].set_title('Detection map')
    axes[1].axis('off')

    axes[2].imshow(ct_arr[best_z],  cmap='gray',   vmin=-200, vmax=300)
    axes[2].imshow(det_arr[best_z], cmap='hot',    vmin=0,    vmax=1,  alpha=0.5)
    axes[2].set_title(f'Overlay  |  score={likelihoods[case_id]:.4f}')
    axes[2].axis('off')

    fig.suptitle(case_id, fontsize=12)
    plt.tight_layout()
    plt.show()

## 9. (Optional) Stage-2 training from scratch

Skip this section if you are using the **pretrained** Stage-2 checkpoints.

The paper uses:
- **Trainer**: `nnUNetTrainerCELossLesionSplit` — cross-entropy loss only (no Dice term)
- **Architecture**: ResU-Net (`nnUNetPlans_v3`)
- **5-fold cross-validation** on the cropped ROI dataset

Prerequisite: prepare Dataset 107 by running nnU-Net dataset conversion on the cropped ROIs  
(`nnUNetv2_plan_and_preprocess -d 107 --verify_dataset_integrity`)

In [ ]:
# ── Only run if you want to train Stage-2 models from scratch ────────────────
TRAIN_STAGE2 = False   # set to True to enable

if TRAIN_STAGE2:
    os.environ['nnUNet_results'] = MODEL_DIR
    os.environ['nnUNet_raw']     = osp.join(REPO_ROOT, 'workspace', 'nnUNet_raw')
    os.environ['nnUNet_preprocessed'] = osp.join(REPO_ROOT, 'workspace', 'nnUNet_preprocessed')

    for fold in range(5):
        train_cmd = [
            'nnUNetv2_train',
            str(STAGE2_TASK),
            '3d_fullres',
            str(fold),
            '-tr',  STAGE2_TRAINER,
            '-p',   STAGE2_PLAN,
        ]
        print(f'Training fold {fold}:', ' '.join(train_cmd))
        subprocess.check_call(train_cmd)
else:
    print('Training skipped (TRAIN_STAGE2=False). Using pretrained checkpoints.')

## 10. Summary report

In [ ]:
with open(osp.join(OUTPUT_DIR, 'pdac-likelihood.json')) as f:
    scores = json.load(f)

print('=' * 55)
print('  PanDx — patient-level PDAC likelihood scores')
print('=' * 55)
for case, s in sorted(scores.items()):
    bar   = '█' * int(s * 40)
    label = 'HIGH' if s >= 0.5 else 'low '
    print(f'  {case:35s}  {s:.4f}  {label}  {bar}')
print('=' * 55)

vals = list(scores.values())
if vals:
    plt.figure(figsize=(max(4, len(vals)*1.5), 4))
    plt.bar(scores.keys(), vals, color=['tomato' if v >= 0.5 else 'steelblue' for v in vals])
    plt.axhline(0.5, color='red', linestyle='--', label='threshold = 0.5')
    plt.ylim(0, 1)
    plt.ylabel('PDAC likelihood')
    plt.title('Patient-level detection scores')
    plt.xticks(rotation=30, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 11. Cleanup intermediate files

Removes the `itm/` working directory (low-res images, cropped images, raw prediction npz files).  
The final detection maps and likelihood JSON in `output/` are preserved.

In [ ]:
CLEANUP = False   # set to True when you are done and want to free disk space

if CLEANUP and osp.exists(WORKING_DIR):
    shutil.rmtree(WORKING_DIR)
    print(f'Removed: {WORKING_DIR}')
else:
    print(f'Cleanup skipped. Intermediate files at: {WORKING_DIR}')